In [41]:
# ============================================================
# MODULE 4.1 — PDF LOGICAL SPLITTING
# Boundary Detection Pipeline
# ============================================================

# ── CELL 1: CÀI ĐẶT ─────────────────────────────────────────
!pip install pymupdf pdfplumber pdf2image pytesseract -q
!apt-get install -y tesseract-ocr tesseract-ocr-vie -q

import fitz, pdfplumber, re, json
import pytesseract
from pdf2image import convert_from_path
from google.colab import drive

print("✅ Setup xong!")

Reading package lists...
Building dependency tree...
Reading state information...
tesseract-ocr is already the newest version (4.1.1-2.1build1).
tesseract-ocr-vie is already the newest version (1:4.00~git30-7274cfa-1.1).
0 upgraded, 0 newly installed, 0 to remove and 1 not upgraded.
✅ Setup xong!


In [42]:
# ── CELL 2: CONFIG ───────────────────────────────────────────
PDF_PATH = '/content/drive/MyDrive/VNDigitizeComprehensiveSystem_Team03/dataset_module4/mix_doc/document_merged.pdf'

# Regex bắt Số văn bản (VD: 913/2025/QĐST-HC, 69/2009/NĐ-CP)
RE_SO_VAN_BAN = r'\d{1,3}/\d{4}/[A-ZĐQĐ][\w\-]+'

# Ground truth để test
GT_BOUNDARIES = {1, 23,29, 32, 35, 44} #
GT_SEPARATORS = {3, 29}

with fitz.open(PDF_PATH) as doc:
    TOTAL_PAGES = len(doc)

print(f"✅ Config xong | {PDF_PATH.split('/')[-1]} | {TOTAL_PAGES} trang")

✅ Config xong | document_merged.pdf | 45 trang


In [43]:
# ── CELL 3: EXTRACT TEXT & FEATURES ────────────────────────
import io
from PIL import Image

def get_text_hybrid_fast(pdf_path: str, page_idx: int) -> str:
    with fitz.open(pdf_path) as doc:
        page = doc[page_idx]
        text = page.get_text("text").strip()

        # Nếu không có text gốc (ảnh scan), chuyển sang OCR tốc độ cao
        if len(text) < 10:
            pix = page.get_pixmap(dpi=150, colorspace=fitz.csGRAY)
            img = Image.frombytes("L", [pix.width, pix.height], pix.samples)
            text = pytesseract.image_to_string(img, lang='vie').strip()

    return text


def extract_features(pdf_path: str, page_idx: int) -> dict:
    # ĐÃ SỬA LỖI: Gọi đúng hàm hybrid OCR
    text = get_text_hybrid_fast(pdf_path, page_idx)
    lines = text.split("\n") if text else []
    n = len(lines)

    # Gom 20% số dòng đầu tiên làm Header (tối thiểu 5 dòng) để định vị
    header = " ".join(l.strip() for l in lines[:max(5, int(n * 0.20))] if l.strip())
    header_up = header.upper()

    # 1. Tìm Quốc hiệu / Tiêu ngữ / Tên cơ quan đặc trưng TẠI HEADER
    has_quoc_hieu = "CỘNG HÒA XÃ HỘI" in header_up
    has_tieu_ngu = "HẠNH PHÚC" in header_up
    has_toa_an = "TÒA ÁN NHÂN DÂN" in header_up

    # 2. Bắt Số văn bản, nhưng CHỈ BẮT Ở HEADER để tránh dính trích dẫn
    so_vb_header = re.findall(RE_SO_VAN_BAN, header)

    return {
        "page_num": page_idx + 1,
        "char_count": len(text),
        "is_blank": len(text) < 30,
        "header": header,
        "has_quoc_hieu": has_quoc_hieu,
        "has_tieu_ngu": has_tieu_ngu,
        "has_toa_an": has_toa_an,
        "so_vb_header": so_vb_header,
        "text_full": text,
    }

# Chạy extract
print("⏳ Đang extract features (sử dụng Hybrid OCR)...\n")
print(f"{'Trang':<7}{'Ký tự':<8}{'Blank':<7}{'Quốc Hiệu':<12}{'Tòa Án':<10}{'Số VB (Header)'}")
print("─" * 75)

all_features = []
for i in range(TOTAL_PAGES):
    f = extract_features(PDF_PATH, i)
    all_features.append(f)

    so_vb_str = f["so_vb_header"][0] if f["so_vb_header"] else "—"
    qh_str = "✅" if f['has_quoc_hieu'] else "—"
    ta_str = "✅" if f['has_toa_an'] else "—"

    print(
        f"  {f['page_num']:<5}"
        f"{f['char_count']:<8}"
        f"{'🔴' if f['is_blank'] else '':<7}"
        f"{qh_str:<12}"
        f"{ta_str:<10}"
        f"{so_vb_str}"
    )

print("─" * 75)
print(f"✅ Extract xong {len(all_features)} trang!")

⏳ Đang extract features (sử dụng Hybrid OCR)...

Trang  Ký tự   Blank  Quốc Hiệu   Tòa Án    Số VB (Header)
───────────────────────────────────────────────────────────────────────────
  1    1812           —           ✅         427/2019/HC-PT
  2    2424           —           —         —
  3    3095           —           —         —
  4    3175           —           —         —
  5    3000           —           —         —
  6    3310           —           —         69/2009/NĐ-CP
  7    3090           —           —         —
  8    3279           —           —         14/2009/TT-BTNMT
  9    2530           —           —         —
  10   2801           —           —         —
  11   2811           —           —         —
  12   2557           —           —         —
  13   3035           —           —         87/2009/QĐ-UBND
  14   2624           —           —         —
  15   2770           —           —         —
  16   3064           —           —         —
  17   2841           —   

In [44]:
# ── CELL 4: BOUNDARY DETECTION ───────────────────────────────
def is_boundary(feat: dict, prev: dict | None) -> tuple[bool, str, float]:
    # 1. Trang đầu tiên luôn là ranh giới
    if prev is None:
        return True, "first_page", 1.0

    # 2. Bỏ qua trang trắng
    if feat["is_blank"]:
        return False, "blank", 0.0

    signals = []
    conf = 0.0

    # 3. LUẬT MẠNH 1: Có Quốc hiệu + Tiêu ngữ ở phần đầu trang
    if feat["has_quoc_hieu"] and feat["has_tieu_ngu"]:
        signals.append("quoc_hieu_tieu_ngu_in_header")
        conf += 1.0

    # 4. LUẬT MẠNH 2: Có cụm "Tòa án" (hoặc cơ quan khác) và có Số văn bản ở Header
    elif feat["has_toa_an"] and feat["so_vb_header"]:
        signals.append(f"toa_an_va_so_vb_in_header:{feat['so_vb_header'][0]}")
        conf += 0.9

    return conf >= 0.8, " + ".join(signals) if signals else "continuation", conf

print("BOUNDARY DETECTION RESULTS")
print("=" * 95)
print(f"{'Trang':<7}{'Predict':<13}{'GT':<13}{'Match':<9}{'Conf':<7}{'Reason'}")
print("─" * 95)

predict_set = set()
for i, feat in enumerate(all_features):
    prev = all_features[i-1] if i > 0 else None
    bdry, reason, conf = is_boundary(feat, prev)
    pn = feat["page_num"]

    if bdry:
        predict_set.add(pn)

    gt_label   = "BOUNDARY" if pn in GT_BOUNDARIES else ("SEPARATOR" if pn in GT_SEPARATORS else "continuation")
    pred_label = "BOUNDARY" if bdry else ("separator" if feat["is_blank"] else "continuation")
    match      = ("✅ HIT" if bdry else "❌ MISS") if pn in GT_BOUNDARIES else \
                 ("⚠️ FP"  if bdry else "") if pn not in GT_SEPARATORS else ""

    if match or pn in GT_BOUNDARIES | GT_SEPARATORS:
        print(f"  {pn:<5}{pred_label:<13}{gt_label:<13}{match:<9}{conf:<7}{reason[:55]}")

print("=" * 95)

BOUNDARY DETECTION RESULTS
Trang  Predict      GT           Match    Conf   Reason
───────────────────────────────────────────────────────────────────────────────────────────────
  1    BOUNDARY     BOUNDARY     ✅ HIT    1.0    first_page
  3    continuation SEPARATOR             0.0    continuation
  23   BOUNDARY     BOUNDARY     ✅ HIT    0.9    toa_an_va_so_vb_in_header:03/2017/DS-ST
  29   BOUNDARY     BOUNDARY     ✅ HIT    1.0    quoc_hieu_tieu_ngu_in_header
  32   BOUNDARY     BOUNDARY     ✅ HIT    1.0    quoc_hieu_tieu_ngu_in_header
  35   BOUNDARY     BOUNDARY     ✅ HIT    1.0    quoc_hieu_tieu_ngu_in_header
  44   BOUNDARY     BOUNDARY     ✅ HIT    1.0    quoc_hieu_tieu_ngu_in_header


In [45]:
# ── CELL 5: METRICS ──────────────────────────────────────────
TP = len(predict_set & GT_BOUNDARIES)
FP = len(predict_set - GT_BOUNDARIES)
FN = len(GT_BOUNDARIES - predict_set)

p  = TP / (TP + FP) if (TP + FP) > 0 else 0
r  = TP / (TP + FN) if (TP + FN) > 0 else 0
f1 = 2*p*r / (p+r)  if (p + r)   > 0 else 0

print("📊 KẾT QUẢ")
print("=" * 50)
print(f"  GT predict    : {sorted(GT_BOUNDARIES)}")
print(f"  Predict       : {sorted(predict_set)}")
print(f"  TP / FP / FN  : {TP} / {FP} / {FN}")
print(f"  Precision     : {p:.3f}")
print(f"  Recall        : {r:.3f}")
print(f"  F1 Score      : {f1:.3f}  {'✅ Đạt' if f1 >= 0.90 else '⚠️ Chưa đạt 0.90'}")
print("=" * 50)

for pn in sorted(GT_BOUNDARIES - predict_set):
    f = all_features[pn-1]
    print(f"\n❌ Miss trang {pn}: Q.Hiệu={f['has_quoc_hieu']} | T.Án={f['has_toa_an']} | Số VB={f['so_vb_header']}")
    print(f"   Header: {f['header'][:120]}")

for pn in sorted(predict_set - GT_BOUNDARIES):
    f = all_features[pn-1]
    print(f"\n⚠️  FP  trang {pn}: Q.Hiệu={f['has_quoc_hieu']} | T.Án={f['has_toa_an']} | Số VB={f['so_vb_header']}")
    print(f"   Header: {f['header'][:120]}")

📊 KẾT QUẢ
  GT predict    : [1, 23, 29, 32, 35, 44]
  Predict       : [1, 23, 29, 32, 35, 44]
  TP / FP / FN  : 6 / 0 / 0
  Precision     : 1.000
  Recall        : 1.000
  F1 Score      : 1.000  ✅ Đạt
